In [1]:
import os
import numpy as np
from scipy import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TimeDistributed,
    Conv2D,
    MaxPooling2D,
    Flatten,
    LSTM,
    Dense,
    Dropout
)

DATASET_PATH = "."

def get_label(filename):

    filename = os.path.splitext(filename)[0]

    parts = filename.split("-")

    gesture = int(parts[1])

    return gesture - 1

max_frames = 0

for file in os.listdir(DATASET_PATH):

    if file.endswith(".mat"):

        cube = io.loadmat(os.path.join(DATASET_PATH,file))["velocity_spectrum_ro"]

        print(file, cube.shape)

        if cube.shape[2] > max_frames:
            max_frames = cube.shape[2]

print("\nMaximum Frames =", max_frames)


def resize_frames(cube,target_frames):

    h,w,t = cube.shape

    if t > target_frames:

        cube = cube[:,:,:target_frames]

    elif t < target_frames:

        new_cube = np.zeros((h,w,target_frames),dtype=np.float32)

        new_cube[:,:,:t] = cube

        cube = new_cube

    return cube


X = []
y = []

for file in os.listdir(DATASET_PATH):

    if file.endswith(".mat"):

        data = io.loadmat(os.path.join(DATASET_PATH,file))

        cube = data["velocity_spectrum_ro"]

        cube = resize_frames(cube,max_frames)

        X.append(cube)

        y.append(get_label(file))

X = np.array(X,dtype=np.float32)

y = np.array(y)

print("\nDataset Shape:",X.shape)
print("Labels Shape :",y.shape)


X = (X-X.mean())/(X.std()+1e-8)


X = np.transpose(X,(0,3,1,2))

X = X[...,np.newaxis]

print("Input Shape =",X.shape)

X_train,X_test,y_train,y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

num_classes = len(np.unique(y))

time_steps = X.shape[1]

height = X.shape[2]

width = X.shape[3]

model = Sequential()

model.add(

    TimeDistributed(

        Conv2D(

            32,

            (3,3),

            activation="relu",

            padding="same"

        ),

        input_shape=(time_steps,height,width,1)

    )

)

model.add(TimeDistributed(MaxPooling2D((2,2))))

model.add(

    TimeDistributed(

        Conv2D(

            64,

            (3,3),

            activation="relu",

            padding="same"

        )

    )

)

model.add(TimeDistributed(MaxPooling2D((2,2))))

model.add(TimeDistributed(Flatten()))

model.add(LSTM(128))

model.add(Dropout(0.5))

model.add(Dense(64,activation="relu"))

model.add(Dense(num_classes,activation="softmax"))

model.summary()


model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


history = model.fit(

    X_train,

    y_train,

    validation_data=(X_test,y_test),

    epochs=30,

    batch_size=8

)


loss,accuracy = model.evaluate(X_test,y_test)

print("\nAccuracy =",accuracy)


pred = model.predict(X_test)

pred = np.argmax(pred,axis=1)

print("\nConfusion Matrix")

print(confusion_matrix(y_test,pred))

print("\nClassification Report")

print(classification_report(y_test,pred))


sample = X_test[0:1]

prediction = model.predict(sample)

gesture = np.argmax(prediction)+1

print("\nPredicted Gesture =",gesture)

print("Actual Gesture =",y_test[0]+1)

model.save("RadarGestureModel.keras")

print("\nModel Saved Successfully")

user1-5-1-4-5.mat (20, 20, 25)
user1-2-2-4-17.mat (20, 20, 17)
user1-1-5-4-7.mat (20, 20, 15)
user1-1-5-1-7.mat (20, 20, 15)
user1-3-4-2-4.mat (20, 20, 12)
user1-5-2-2-19.mat (20, 20, 26)
user1-3-1-1-18.mat (20, 20, 14)
user1-5-2-5-4.mat (20, 20, 24)
user1-2-3-2-14.mat (20, 20, 18)
user1-3-3-5-19.mat (20, 20, 13)
user1-1-1-1-11.mat (20, 20, 15)
user1-2-1-3-20.mat (20, 20, 18)
user1-2-4-3-10.mat (20, 20, 15)
user1-1-2-4-6.mat (20, 20, 16)
user1-5-2-5-9.mat (20, 20, 24)
user1-4-2-1-3.mat (20, 20, 14)
user1-3-1-5-13.mat (20, 20, 11)
user1-5-3-3-7.mat (20, 20, 24)
user1-4-3-5-6.mat (20, 20, 19)
user1-3-2-1-17.mat (20, 20, 12)
user1-5-2-3-18.mat (20, 20, 28)
user1-3-3-5-17.mat (20, 20, 12)
user1-3-1-2-4.mat (20, 20, 11)
user1-1-2-4-5.mat (20, 20, 16)
user1-3-4-3-17.mat (20, 20, 11)
user1-4-4-5-1.mat (20, 20, 17)
user1-4-3-2-9.mat (20, 20, 19)
user1-5-2-1-13.mat (20, 20, 25)
user1-5-2-2-2.mat (20, 20, 22)
user1-3-2-1-12.mat (20, 20, 12)
user1-2-2-1-9.mat (20, 20, 20)
user1-4-3-3-2.mat (20, 2

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 28, 20, 20, 32) │           320 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 28, 10, 10, 32) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 28, 10, 10, 64) │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 28, 5, 5, 64)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 28, 1600)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       885,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 912,710 (3.48 MB)

 Trainable params: 912,710 (3.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 188s 772ms/step - accuracy: 0.5931 - loss: 0.9618 - val_accuracy: 0.6616 - val_loss: 0.7179
Epoch 2/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 159s 693ms/step - accuracy: 0.6832 - loss: 0.7524 - val_accuracy: 0.7358 - val_loss: 0.6387
Epoch 3/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 154s 671ms/step - accuracy: 0.7340 - loss: 0.6395 - val_accuracy: 0.7817 - val_loss: 0.5418
Epoch 4/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 201s 667ms/step - accuracy: 0.7854 - loss: 0.5491 - val_accuracy: 0.8231 - val_loss: 0.4947
Epoch 5/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 163s 712ms/step - accuracy: 0.8034 - loss: 0.5081 - val_accuracy: 0.8100 - val_loss: 0.4736
Epoch 6/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 155s 678ms/step - accuracy: 0.8247 - loss: 0.4531 - val_accuracy: 0.8406 - val_loss: 0.3978
Epoch 7/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 172s 546ms/step - accuracy: 0.8607 - loss: 0.3816 - val_accuracy: 0.8231 - val_loss: 0.4287
Epoch 8/30
229/229 ━━━━━━━━━━━━━━━━━━━━ 140s 539ms/step - accuracy: 0.8717 -